In [123]:
from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd
import numpy as np
from langchain_core.documents import Document
import re
from rank_bm25 import BM25Okapi
import pickle

In [124]:
DATA_DIR = Path("data")
CATEGORY = "Kindle_Store"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_FILE = DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE    = DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"
OUTPUT_FILE  = DATA_DIR / f"{CATEGORY}_merged.parquet"

In [125]:
c2 = duckdb.connect()

In [126]:
head_reviews = c2.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()
head_reviews

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,excellent writer reminds me of Clive Cussler,GRUMLEY is on par with Clive Cussler and his D...,[],B00LXRJICK,B00LXRJICK,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1427541413000,0,False
1,3.0,Alright book,The book Fade was not my favorite but was a go...,[],B073DFP8VC,B073DFP8VC,AHGTHCERTEZUXNBLJ5SWHK2CDLXA,1504226946142,0,True
2,5.0,Hats off to Fern Michaels for all her great ac...,I have been a fan of this author for many year...,[],B07QVH25KX,B07QVH25KX,AHFY2QSS6PK5MHSYZFI6TXUYNPLQ,1644883955777,0,True
3,5.0,Great read,This book is more than just about a dog and a ...,[],B004Y1NWQU,B004Y1NWQU,AHFY2QSS6PK5MHSYZFI6TXUYNPLQ,1363027885000,0,False
4,5.0,Add to legend f Arthur,Good twist on the ledgen of King<br />Arthur !...,[],B08M993CNC,B08M993CNC,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1637557512064,0,True


In [127]:
head_meta = c2.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

,main_category,title,subtitle,author,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Buy a Kindle,The Palace (Chateau Book 4),Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.7,970,[A line was drawn in the sand.I made my choice...,[],0.00,[{'large': 'https://m.media-amazon.com/images/...,[],Penelope Sky (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Romance]","{'Publisher': 'Hartwick Publishing (May 25, 20...",B08XPZPFY4,None
1,Buy a Kindle,Microsoft PowerPoint 2016 2013 2010 2007 Tips ...,[Print Replica] Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.3,35,"[Paperback versions are also available, includ...","[From the Author, Amelia Griggs is an Instruct...",0.00,[{'large': 'https://m.media-amazon.com/images/...,[],Amelia Griggs (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Computers & Tech...","{'Publisher': None, 'Publication date': 'June ...",B07DH1LF1K,None
2,Buy a Kindle,Ill Wind (Anna Pigeon Mysteries Book 3),Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.4,1628,"[Lately, visitors to Mesa Verde have been brin...","[From Publishers Weekly, Barr lands another su...",7.99,[{'large': 'https://m.media-amazon.com/images/...,[],Nevada Barr (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Mystery, Thrille...",{'Publisher': 'Berkley; Reissue edition (March...,B0022Q8CTQ,None
3,Buy a Kindle,30 Healthy Easy Quick Lentil Recipes (Brad Arm...,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,3.8,47,"[If improved health you are seeking, look no f...",[],0.00,[{'large': 'https://m.media-amazon.com/images/...,[],Brad Armstrong (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Health, Fitness ...","{'Publisher': 'Brad Armstrong (March 10, 2013)...",B00BS56MLC,None
4,Buy a Kindle,The Road Home,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.5,475,"[In one of Jim Harrison’s greatest works, five...","[Review, ""A graceful novel...To read this book...",10.44,[{'large': 'https://m.media-amazon.com/images/...,[],Jim Harrison (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Literature & Fic...",{'Publisher': 'Atlantic Monthly Press (Decembe...,B00155EZRS,None


In [128]:
c2.execute(f"""
    COPY ( SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 100000)
    TO '../data/raw/reviews_raw.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [129]:
c2.execute(f"""
      COPY (SELECT * FROM read_json_auto('{META_URL}') LIMIT 50000)
      TO '../data/raw/meta_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """)

In [130]:
reviews_df = c2.execute(f"SELECT * FROM read_parquet('../data/raw/reviews_raw.parquet')").df()
meta_df = c2.execute(f"SELECT * FROM read_parquet('../data/raw/meta_raw.parquet')").df()

In [131]:
reviews_df.describe(), reviews_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   rating             100000 non-null  float64
 1   title              100000 non-null  str    
 2   text               100000 non-null  str    
 3   images             100000 non-null  object 
 4   asin               100000 non-null  str    
 5   parent_asin        100000 non-null  str    
 6   user_id            100000 non-null  str    
 7   timestamp          100000 non-null  int64  
 8   helpful_vote       100000 non-null  int64  
 9   verified_purchase  100000 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(1), str(5)
memory usage: 67.9+ MB


(              rating     timestamp   helpful_vote
 count  100000.000000  1.000000e+05  100000.000000
 mean        4.342320  1.522615e+12       1.059320
 std         0.983284  9.102178e+10       7.386213
 min         1.000000  1.127106e+12       0.000000
 25%         4.000000  1.450552e+12       0.000000
 50%         5.000000  1.526708e+12       0.000000
 75%         5.000000  1.598912e+12       1.000000
 max         5.000000  1.679239e+12     888.000000,
 None)

In [132]:

meta_df.describe(), meta_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    49998 non-null  str    
 1   title            50000 non-null  str    
 2   subtitle         49084 non-null  str    
 3   author           42206 non-null  object 
 4   average_rating   50000 non-null  float64
 5   rating_number    50000 non-null  int64  
 6   features         50000 non-null  object 
 7   description      50000 non-null  object 
 8   price            47257 non-null  float64
 9   images           50000 non-null  object 
 10  videos           50000 non-null  object 
 11  store            50000 non-null  str    
 12  categories       50000 non-null  object 
 13  details          50000 non-null  object 
 14  parent_asin      50000 non-null  str    
 15  bought_together  0 non-null      object 
dtypes: float64(2), int64(1), object(8), str(5)
memory usage: 13.1+ MB


(       average_rating  rating_number         price
 count    50000.000000   50000.000000  47257.000000
 mean         4.307856     424.006680      3.857522
 std          0.513857    2518.897923      8.333157
 min          1.000000       1.000000      0.000000
 25%          4.100000      12.000000      0.000000
 50%          4.400000      45.000000      0.990000
 75%          4.600000     198.000000      4.990000
 max          5.000000  138549.000000    982.990000,
 None)

In [133]:
# Select only the required columns and perform an INNER JOIN to merge reviews and meta.
# NOTE: A LEFT JOIN is avoided to prevent introducing many missing values.

c2.execute("""
    COPY (
         SELECT r.parent_asin, r.rating, r.title AS review_title, r.text AS review_text, r.verified_purchase,
            m.title AS product_title, m.average_rating, m.main_category, m.description, m.features
        FROM read_parquet('../data/raw/reviews_raw.parquet') r
        INNER JOIN read_parquet('../data/raw/meta_raw.parquet') m USING (parent_asin)
    )
    TO '../data/processed/merged.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [134]:
merged_df = c2.execute(f"SELECT * FROM read_parquet('../data/processed/merged.parquet')").df()

In [135]:
merged_df.head(5)

,parent_asin,rating,review_title,review_text,verified_purchase,product_title,average_rating,main_category,description,features
0,B07FQTND9B,4.0,Good start,Interesting n intertaining concept. Looking fo...,False,Thrive: A Post-Apocalyptic Alien Survival Seri...,4.2,Buy a Kindle,"[About the Author, Jonathan Yanez is the autho...","[Would you choose peace over truth?, How long ..."
1,B00IQOC2DU,5.0,No easy solution,This story has given me incites into a very di...,True,"Spare Parts: Four Undocumented Teenagers, One ...",4.5,Buy a Kindle,"[Amazon.com Review, An Amazon Best Book of the...","[Joshua Davis's, Spare Parts, --now a major mo..."
2,B01C8KB83M,5.0,Rouse the Ross,Very entertaining. Was still looking for the V...,False,De Oppresso Liber (On Silver Wings Book 6),4.5,Buy a Kindle,"[About the Author, Evan Currie is the bestsell...","[The war may be over, but the fighting continu..."
3,B003GWX8SK,2.0,A little disappointed,"When I picked this book up, I really wanted to...",True,Ender's Shadow (The Shadow Saga Book 1),4.7,Buy a Kindle,"[Review, ""As a maker of visions and a creator ...",[Orson Scott Card brings us back to the very b...
4,B00QMUBB0K,4.0,Some great tips on getting healthy,Some great tips on getting healthy. Great star...,True,Lose Weight Fast: Over 50 Incredible Weight Lo...,3.6,Buy a Kindle,[],"[Are You Sick Of Being Overweight?, This book ..."


In [136]:
merged_df.info(), merged_df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 3781 entries, 0 to 3780
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   parent_asin        3781 non-null   str    
 1   rating             3781 non-null   float64
 2   review_title       3781 non-null   str    
 3   review_text        3781 non-null   str    
 4   verified_purchase  3781 non-null   bool   
 5   product_title      3781 non-null   str    
 6   average_rating     3781 non-null   float64
 7   main_category      3781 non-null   str    
 8   description        3781 non-null   object 
 9   features           3781 non-null   object 
dtypes: bool(1), float64(2), object(2), str(5)
memory usage: 2.6+ MB


(None,
             rating  average_rating
 count  3781.000000     3781.000000
 mean      4.357048        4.362602
 std       0.953965        0.304269
 min       1.000000        1.000000
 25%       4.000000        4.200000
 50%       5.000000        4.400000
 75%       5.000000        4.600000
 max       5.000000        5.000000)

In [137]:
merged_df.isna().mean().sort_values(ascending=False)

parent_asin          0.0
rating               0.0
review_title         0.0
review_text          0.0
verified_purchase    0.0
product_title        0.0
average_rating       0.0
main_category        0.0
description          0.0
features             0.0
dtype: float64

In [138]:

# Function to convert a list of strings into a single string

def to_text(x):
    if x is pd.NA:
        return ""
    if isinstance(x, np.ndarray):
        return " ".join(map(str, x))
    return str(x)

merged_df["features"] = merged_df["features"].apply(to_text)
merged_df["description"] = merged_df["description"].apply(to_text)

In [139]:
merged_df.head(3)

,parent_asin,rating,review_title,review_text,verified_purchase,product_title,average_rating,main_category,description,features
0,B07FQTND9B,4.0,Good start,Interesting n intertaining concept. Looking fo...,False,Thrive: A Post-Apocalyptic Alien Survival Seri...,4.2,Buy a Kindle,About the Author Jonathan Yanez is the author ...,Would you choose peace over truth? How long co...
1,B00IQOC2DU,5.0,No easy solution,This story has given me incites into a very di...,True,"Spare Parts: Four Undocumented Teenagers, One ...",4.5,Buy a Kindle,Amazon.com Review An Amazon Best Book of the M...,Joshua Davis's Spare Parts --now a major motio...
2,B01C8KB83M,5.0,Rouse the Ross,Very entertaining. Was still looking for the V...,False,De Oppresso Liber (On Silver Wings Book 6),4.5,Buy a Kindle,About the Author Evan Currie is the bestsellin...,"The war may be over, but the fighting continue..."


In [140]:
# From the selected columns, investigate which ones are meaningful based on their number of unique values
merged_df.nunique()

parent_asin          2823
rating                  5
review_title         3237
review_text          3742
verified_purchase       2
product_title        2822
average_rating         30
main_category           2
description          1655
features             2792
dtype: int64

In [141]:
# Create the document column by combining the fields that provide the most context
merged_df["document"] = (
    "Product: " + merged_df["product_title"].fillna("") + ". "
    + "Description: " + merged_df["description"].fillna("") + ". "
    + "Features: " + merged_df["features"].fillna("") + ". "
    + "Review Title: " + merged_df["review_title"].fillna("") + ". "
    + "Review: " + merged_df["review_text"].fillna("")
)

In [142]:
merged_df['document'][1]

"Product: Spare Parts: Four Undocumented Teenagers, One Ugly Robot, and the Battle for the American Dream. Description: Amazon.com Review An Amazon Best Book of the Month, December 2014: Spare Parts is the fantastic story of four Mexican-American teenagers struggling to find their place. An unlikely robotics competition becomes the focus of the narrative, but the story covers a lot of ground. By describing how these teens came together, author Joshua Davis gives us a succinct history of immigration and a micro-lesson in Arizona politics. It all leads to the a scene in a pool in Santa Barbara, CA—with each team member realizing how they fit on the team, and in their adopted homeland. – Amy Huff --This text refers to an alternate kindle_edition edition. From the Inside Flap In 2004, four Latino teenagers arrived at a national underwater robotics championship at the University of California, Santa Barbara. Oscar, Lorenzo, Cristian, and Luis were all born in Mexico but raised in Phoenix, A



## Text preprocessing decisions

To prepare the data for retrieval, we applied the following preprocessing steps to preserve useful context while making the text easier to search.

- We merged reviews with metadata using an INNER join on parent_asin so that every document contains both review text and product information.

- We kept informative text fields such as product_title, review_title, review_text, description, and features.

- We removed non-meaningful or non-textual fields such as parent_asin, rating, verified_purchase, and main_category from the final document text.

- The description and features columns originally stored array values, so we converted them into single strings, and missing values were replaced with empty strings.

- We created a column named document which combines product metadata and review text using the following columns: Product, Description, Features, Review Title, and Review.

The preprocessing implemented is initial, and more retrieval-specific preprocessing such as lowercasing, tokenization, punctuation removal, and optional stopword removal will be applied in the upcoming stages.

## Preparing data for retrieval (LangChain format)

For retrieval stages (BM25 and semantic search), we convert each row into LangChain Document objects. This separates the searchable text `page_content' from the metadata.

In [143]:
# Convert each row into LangChain Document format.
# page_content stores searchable text, and metadata keeps identifiers and attributes.

langchain_docs = []

for _, row in merged_df.iterrows():
    doc = Document(
        page_content=row["document"],
        metadata={
            "parent_asin": row["parent_asin"],
            "product_title": row["product_title"],
            "rating": row["rating"],
            "verified_purchase": row["verified_purchase"],
            "average_rating": row["average_rating"],
            "main_category": row["main_category"],
        }
    )
    langchain_docs.append(doc)

In [144]:
print(langchain_docs[0].page_content)
print(langchain_docs[0].metadata)

Product: Thrive: A Post-Apocalyptic Alien Survival Series (The Pandora Experiment Book 1). Description: About the Author Jonathan Yanez is the author of dozens of science fiction and fantasy novels. He's the winner of a Jack London Award for his contributions to literacy. Two of his series have been optioned for film. His latest work is currently under development being adapted into a mobile game. You can find out more about him by visiting www.jonathan-yanez.com --This text refers to an out of print or unavailable edition of this title.. Features: Would you choose peace over truth? How long could that peace last? The citizens enjoy the peace of their perfectly planned city. But when she notices cracks in the facade, Jordan’s waking dreams turn to nightmares. She thought she had received the promotion of a lifetime; she was wrong. The promotion only brings her closer to terrifying truths of the administration. Secrets told to keep the citizens safe turn out to be lies to keep them in l

## BM25

In [145]:

# Function for lowercase, remove punctuation and whitespace tokenization.

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return text.split()

In [146]:
# Create the corpus and tokenize it in preparation for BM25
corpus = merged_df["document"].tolist()

tokenized_corpus = []

for document in corpus:

    tokens = tokenize(document)
    tokenized_corpus.append(tokens)

In [147]:
tokenized_corpus[1][:5]

['product', 'spare', 'parts', 'four', 'undocumented']

In [163]:
def bm25_search(query, bm25, df, top_k=5):

    tokenized_query = tokenize( query)

    scores = bm25.get_scores( tokenized_query)

    sorted_indices = np.argsort(scores)
    top_indices = sorted_indices[-top_k:]
    top_indices = top_indices[::-1]
    
    results = df.iloc[top_indices].copy()
    
    results["bm25_score"] = scores[top_indices]
    
    return results[[
        "product_title",
        "review_title",
        "review_text",
        "bm25_score"
    ]]

In [164]:
bm25_model = BM25Okapi(tokenized_corpus)

query = "weight loss book"
bm25_ex1= bm25_search(query, bm25_model, merged_df, top_k=5)
bm25_ex1

,product_title,review_title,review_text,bm25_score
4,Lose Weight Fast: Over 50 Incredible Weight Lo...,Some great tips on getting healthy,Some great tips on getting healthy. Great star...,16.573258
829,HYPOTHYROIDISM DIET ~ The secrets to your thyr...,Delicious Lentil Recipe,If you are struggling to make any progress in ...,16.236909
1567,Fat Fast Cookbook: 50 Easy Recipes to Jump Sta...,Four Stars,"This is a good book, interesting plan for maki...",15.360669
745,Fat Fast Cookbook: 50 Easy Recipes to Jump Sta...,Another Favorite Cookbook By My Favorite Cookb...,I Dana Carpender has made my life as a diabeti...,15.263279
2304,Yoga For Beginners: An Easy Yoga Guide To Reli...,For beginners?,"The text may be for beginners, encouraging yet...",15.157566


In [165]:
query = "data science "
bm25_ex2= bm25_search(query, bm25_model, merged_df, top_k=5)
bm25_ex2

,product_title,review_title,review_text,bm25_score
3610,Learning JavaScript Data Structures and Algori...,but 85% of the book is a great resource. Factu...,"There were some errors in the book, but 85% of...",10.559256
3612,Mastering JavaScript Object-Oriented Programming,Five Stars,A great read for getting into the OO sides of ...,9.352579
2160,Obsolete Theorem (Across Horizons Book 1),So close...,"As to be expected, this really is a great read...",8.820226
1370,Obsolete Theorem (Across Horizons Book 1),Is Time Travel Possible?,In this story it is possible to take a giant s...,8.771618
3487,The Smart Enough City: Putting Technology in I...,Interesting criticism of applying big tech to ...,The book criticizes big tech companies that cl...,8.702343


In [166]:
query = "scary horror books"
bm25_ex3= bm25_search(query, bm25_model, merged_df, top_k=5)
bm25_ex3

,product_title,review_title,review_text,bm25_score
3474,Molters (The Molting Book 1),I love Zombie books,and this one called Molters was in my realm of...,13.386073
2812,Gretel: A Horror Thriller (Gretel Book One),Read This,Great book!,12.768473
2046,Ararat: A Novel,Ok super easy rainy day book,I was drawn in by the sample and couldn't wait...,11.781110
2991,Rend the Dark,"High suspense fantasy with dark, monster theme...","Book Description, My Take: The village of Grov...",10.210678
3054,Halloween Children's 1: Whistling Ghost,There's a whistler in the house.,A brother and sister (no names) go trick or tr...,10.153170


In [167]:
# Save the tokenized corpus and BM25 index so they can be reused later without rebuilding

Path("../data/processed").mkdir(parents=True, exist_ok=True)

# Save tokenized corpus
with open("../data/processed/tokenized_corpus.pkl", "wb") as f:
    pickle.dump(tokenized_corpus, f)

# Save BM25 index
with open("../data/processed/bm25.pkl", "wb") as f:
    pickle.dump(bm25_model, f)